# Week 01 — Streaming and missing metrics

This notebook measures streaming TTFT, inter-token timing, warm-request latency, and writes a machine-readable report. It assumes the model setup cells from `practical-baseline.ipynb` have been run in this notebook too.

In [6]:
import json, platform, statistics, threading, time
from pathlib import Path
from queue import Queue
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation.streamers import BaseStreamer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float16 if DEVICE in ('mps', 'cuda') else torch.float32

def synchronize_device():
    if DEVICE == 'cuda': torch.cuda.synchronize()
    elif DEVICE == 'mps': torch.mps.synchronize()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'python': '3.12.10', 'torch': '2.13.0', 'transformers': '5.16.1', 'device': 'mps'}


In [7]:
def make_inputs(text):
    messages = [{'role': 'user', 'content': text}]
    encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    return {key: value.to(DEVICE) for key, value in encoded.items()}

class TokenTimingStreamer(BaseStreamer):
    def __init__(self):
        self.queue = Queue()
        self.skip_prompt = True

    def put(self, value):
        if value.ndim == 1: value = value.unsqueeze(0)
        if self.skip_prompt:
            self.skip_prompt = False
            return
        for token_id in value[0].tolist():
            self.queue.put(('token', int(token_id), time.perf_counter()))

    def end(self):
        self.queue.put(('end', None, time.perf_counter()))

    def __iter__(self):
        while True:
            item = self.queue.get()
            if item[0] == 'end': return
            yield item

In [8]:
def stream_with_timings(text, max_new_tokens=64):
    inputs = make_inputs(text)
    streamer = TokenTimingStreamer()
    kwargs = dict(inputs, streamer=streamer, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    synchronize_device()
    started = time.perf_counter()
    thread = threading.Thread(target=model.generate, kwargs=kwargs)
    thread.start()
    token_events = []
    print('stream: ', end='', flush=True)
    for event in streamer:
        token_events.append(event)
        token_text = tokenizer.decode([event[1]], skip_special_tokens=True)
        print(token_text, end='', flush=True)
    print()
    thread.join()
    synchronize_device()
    finished = time.perf_counter()
    token_ids = [event[1] for event in token_events]
    timestamps = [event[2] for event in token_events]
    ttft = timestamps[0] - started if timestamps else None
    intervals = [b - a for a, b in zip(timestamps, timestamps[1:])]
    return {
        'text': tokenizer.decode(token_ids, skip_special_tokens=True),
        'prompt_tokens': int(inputs['input_ids'].shape[-1]),
        'output_tokens': len(token_ids),
        'ttft_seconds': ttft,
        'inter_token_latency_seconds': intervals,
        'total_latency_seconds': finished - started,
        'effective_output_tokens_per_second': len(token_ids) / (finished - started) if token_ids else None,
    }

stream_result = stream_with_timings('Explain prefill and decode in simple terms.')
print({k: v for k, v in stream_result.items() if k != 'text' and k != 'inter_token_latency_seconds'})
print('text:', stream_result['text'])

stream: Certainly! Let's break down the concepts of "prefill" and "decode" in a way that is easy to understand.

### Prefill

**Prefill** refers to the process of preparing or setting up something before it needs to be used. It involves creating a template or blueprint for what you want to include
{'prompt_tokens': 39, 'output_tokens': 64, 'ttft_seconds': 0.1085566249967087, 'total_latency_seconds': 1.2679389159966377, 'effective_output_tokens_per_second': 50.475617707256895}
text: Certainly! Let's break down the concepts of "prefill" and "decode" in a way that is easy to understand.

### Prefill

**Prefill** refers to the process of preparing or setting up something before it needs to be used. It involves creating a template or blueprint for what you want to include


## Warm-request comparison

The model was loaded before timing. The first request below is a first-request measurement for an already-loaded model, not full process cold start. Full cold start includes Python process startup, model loading, and model/device initialization.

In [9]:
prompt = 'Explain caching in one sentence.'
warm_results = []
for repeat in range(5):
    print(f'\nrun {repeat + 1}/5')
    result = stream_with_timings(prompt, max_new_tokens=64)
    result.update({'prompt_name': 'short', 'repeat': repeat + 1, 'model_id': MODEL_ID, 'device': DEVICE})
    warm_results.append(result)

ttfts = [r['ttft_seconds'] for r in warm_results if r['ttft_seconds'] is not None]
totals = [r['total_latency_seconds'] for r in warm_results]
itls = [x for r in warm_results for x in r['inter_token_latency_seconds']]
print({'ttft_median': statistics.median(ttfts), 'total_median': statistics.median(totals), 'itl_median': statistics.median(itls), 'runs': len(warm_results)})

stream: Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.
stream: Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.
stream: Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.
stream: Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.
stream: Caching is the process of storing frequently accessed data in memory to reduce the number of requests made to the server, improving performance and reducing load on the server.
{'ttft_median': 0.028593084003659897, 'total_median': 0.599556749999465, 'i

In [10]:
report = {
    'model_id': MODEL_ID,
    'device': DEVICE,
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'generation': {'max_new_tokens': 64, 'do_sample': False, 'measured_runs': 5},
    'streaming_results': warm_results,
}
cwd = Path.cwd()
week1_dir = cwd / 'week-01-baseline-server' if (cwd / 'week-01-baseline-server').is_dir() else cwd
week1_dir.mkdir(parents=True, exist_ok=True)
output_path = week1_dir / 'streaming-metrics.json'
output_path.write_text(json.dumps(report, indent=2))
print(f'wrote {output_path.resolve()}')

wrote /Users/myatkaung/Desktop/production-llm-inference-lab/week-01-baseline-server/streaming-metrics.json


## Interpretation checklist

- TTFT includes prompt processing/prefill and time until the first generated token reaches the streamer.
- Inter-token latency measures the time between generated token events during decode.
- Total latency runs from before generation until generation completes.
- These measurements still do not include HTTP, network, queue, or gateway overhead.
- The next implementation step is the Week 01 HTTP model server, followed by Week 02 gateway contract tests.